# El RAG sintético: el permiso antes o después de recuperar

Cuaderno de lectura de la medición `rag-sintetico/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda el artículo [un-rag-que-aguante-una-inspeccion](https://manpla.net/posts/un-rag-que-aguante-una-inspeccion/). Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/rag-sintetico/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
c = leer("consultas.csv"); res = json.loads(texto("resumen.json"))
print(res["motor"], "·", res["corpus"], "fragmentos ·", res["n_consultas"], "consultas · k =", res["k"])
print("post-filtrado: media de", res["post_exp_media"], "ajenos por consulta,", res["post_con_exposicion"], "consultas con exposición,", res["post_vacias"], "vacías · pre-filtrado:", res["pre_exp_total"], "ajenos")
c[["consulta","post_utiles","post_expuestos","post_vacia","pre_utiles","pre_expuestos"]].head(15)

## Una figura

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.hist(c.post_expuestos, bins=range(0, 12), align="left", rwidth=.8); a.set_xlabel("fragmentos ajenos leídos por consulta (k = 10)"); a.set_ylabel("consultas"); a.set_title("permiso después de recuperar")
b.hist(c.pre_expuestos, bins=range(0, 12), align="left", rwidth=.8, color="C2"); b.set_xlabel("fragmentos ajenos leídos por consulta"); b.set_title("permiso dentro de la consulta")
plt.tight_layout()